In [57]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, from_json, current_timestamp, trim
from pyspark.sql.types import StructType, StringType, StructField
from delta import configure_spark_with_delta_pip

In [58]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-core2.12:3.4.1 pyspark-shell"

In [ ]:
# Crear sesión de Spark con Delta
builder = (
    SparkSession.builder
    .appName("Delta Lake")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:3.4.1") 
)

spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1"]).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

ConnectionRefusedError: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión

In [ ]:
import time

from delta.tables import DeltaTable

# Define rutas absolutas
checkpoint_path = "C:/Users/sandr/TFM-develop/checkpoints"
delta_path = "C:/Users/sandr/TFM-develop/delta_lake"


checkpoint_clean_path = "C:/Users/sandr/TFM-develop/checkpoints/clean"
clean_path = "C:/Users/sandr/TFM-develop/delta_lake/clean"


# Esquema común para todas las fuentes
schema = StructType([
    StructField("source", StringType(), True),
    StructField("sport", StringType(), True),
    StructField("event", StringType(), True),
    StructField("text", StringType(), True),
    StructField("created_at", StringType(), True)
])

df = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "youtube-comments") \
        .load()

# Parsear JSON
parsed = df.select(from_json(col("value").cast("string"), schema).alias("data")).select("data.text")

spark.conf.set("spark.hadoop.io.native.lib.available", "false")

# Escribir en Delta Lake
query = df.writeStream.format("delta").save(delta_path)

time.sleep(20)
query.stop()

Py4JJavaError: An error occurred while calling o611.save.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at io.delta.storage.HadoopFileSystemLogStore.listFrom(HadoopFileSystemLogStore.java:59)
	at org.apache.spark.sql.delta.storage.LogStoreAdaptor.listFrom(LogStore.scala:452)
	at org.apache.spark.sql.delta.storage.DelegatingLogStore.

In [ ]:
df_stream = spark.readStream.format("delta").load(delta_path)

df_clean = df_stream.dropna()\
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("text", trim(col("text"))) \
    .withColumn("text", lower(col("text"))) \
    .withColumn("text", regexp_replace(col("text"), r"\bi m\b", "i'm")) \
    .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z0-9\s']", "")) \
    .dropDuplicates(["source","sport" ,"event","text"])

# Guardar clean
df_clean.writeStream.format("csv").mode("append").save(clean_path)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: C:/Users/sandr/TFM-develop/delta_lake.

In [ ]:

# schema = StructType() \
#     .add("source", StringType()) \
#     .add("sport", StringType()) \
#     .add("event", StringType()) \
#     .add("text", StringType()) \
#     .add("created_at", StringType())

# Función genérica de lectura, limpieza y escritura
# def process_stream(topic, output_path, checkpoint_path):
#     # Leer de Kafka
#     df = spark.readStream \
#         .format("kafka") \
#         .option("kafka.bootstrap.servers", "localhost:9092") \
#         .option("subscribe", topic) \
#         .load()

#     # Parsear JSON
#     parsed = df.selectExpr("CAST(value AS STRING) as json") \
#         .select(from_json(col("json"), schema).alias("data")) \
#         .select("data.*")

#     # Limpieza básica del texto
#     # cleaned = parsed.dropna()\
#     # .withColumn("ingestion_time", current_timestamp()) \
#     # .withColumn("text", trim(col("text"))) \
#     # .withColumn("text", lower(col("text"))) \
#     # .withColumn("text", regexp_replace(col("text"), r"\bi m\b", "i'm")) \
#     # .withColumn("text", regexp_replace(col("text"), r"[^a-zA-Z0-9\s']", ""))  
#     # mantener letras, espacios, apóstrofes

#     # Escribir en Delta Lake
#     query = parsed.writeStream \
#         .format("delta") \
#         .option("checkpointLocation", checkpoint_path) \
#         .outputMode("append") \
#         .start(output_path)
    
#     print("Reddit comments written in Delta Lake")

#     return query

#landing_path = "./../../../delta_lake/"
#cp_path = "./../../../delta_lake/checkpoint"

# Lanzar streams para cada fuente
# reddit_query = process_stream(
#     topic="reddit-comments",
#     output_path=f"{landing_path}/reddit/landing",
#     checkpoint_path=f"{cp_path}/reddit"
# )
